# solving fits in folder

## 필요한 모듈

이 프로젝트를 위해서는 아래의 모듈이 필요하다. 

> numpy, pandas, matplotlib, scipy, astropy, photutils, ccdproc, version_information

### 모듈 설치

1. 콘솔 창에서 모듈을 설치할 때는 아래와 같은 형식으로 입력하면 된다.

>pip install module_name==version

>conda install module_name=version

2. 주피터 노트북(코랩 포함)에 설치 할 때는 아래의 셀을 실행해서 실행되지 않은 모듈을 설치할 수 있다. (pip 기준) 만약 아나콘다 환경을 사용한다면 7행을 콘다 설치 명령어에 맞게 수정하면 된다.

### 모듈 버전 확인

아래 셀을 실행하면 이 노트북을 실행한 파이썬 및 관련 모듈의 버전을 확인할 수 있다.

In [1]:
import importlib, sys, subprocess
packages = "numpy, pandas, matplotlib, scipy, astropy, photutils, ccdproc, version_information" # required modules
pkgs = packages.split(", ")
for pkg in pkgs :
    if not importlib.util.find_spec(pkg):
        #print(f"**** module {pkg} is not installed")
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
    else: 
        print(f"**** module {pkg} is installed")

%load_ext version_information
import time
now = time.strftime("%Y-%m-%d %H:%M:%S (%Z = GMT%z)")
print(f"This notebook was generated at {now} ")

vv = %version_information {packages}
for i, pkg in enumerate(vv.packages):
    print(f"{i} {pkg[0]:10s} {pkg[1]:s}")

**** module numpy is installed
**** module pandas is installed
**** module matplotlib is installed
**** module scipy is installed
**** module astropy is installed
**** module photutils is installed
**** module ccdproc is installed
**** module version_information is installed
This notebook was generated at 2023-10-13 06:47:45 (KST = GMT+0900) 
0 Python     3.11.5 64bit [GCC 11.2.0]
1 IPython    8.15.0
2 OS         Linux 5.15.0 86 generic x86_64 with glibc2.31
3 numpy      1.26.0
4 pandas     2.0.3
5 matplotlib 3.7.2
6 scipy      1.11.1
7 astropy    5.2.1
8 photutils  1.6.0
9 ccdproc    2.4.0
10 version_information 1.0.4


### import modules

In [2]:
from glob import glob
from pathlib import Path
import os
import numpy as np

import matplotlib.pyplot as plt
from astropy.io import fits
import astropy.units as u
from astropy.stats import sigma_clip
from ccdproc import combine, ccd_process, CCDData
from datetime import datetime, timedelta

import ysfitsutilpy as yfu
import ysphotutilpy as ypu
import ysvisutilpy as yvu

import _astro_utilities
import _Python_utilities

plt.rcParams.update({'figure.max_open_warning': 0})

In [3]:
#%%
BASEDIR = Path("/mnt/Rdata/OBS_data") 

DOINGDIR = Path(BASEDIR / "ccd_test_folder")
#DOINGDIR = Path(BASEDIR/ "CCD_new_files1")

DOINGDIRs = sorted(_Python_utilities.getFullnameListOfallsubDirs(DOINGDIR))
#print ("DOINGDIRs: ", format(DOINGDIRs))
print ("len(DOINGDIRs): ", format(len(DOINGDIRs)))
#######################################################

len(DOINGDIRs):  1


In [4]:
for DOINGDIR in DOINGDIRs[:1] :
    DOINGDIR = Path(DOINGDIR)
    print("DOINGDIR", DOINGDIR)
    fits_in_dir = sorted(list(DOINGDIR.glob('*.fit*')))
    #print("fits_in_dir", fits_in_dir)
    print("len(fits_in_dir)", len(fits_in_dir))

    if len(fits_in_dir) == 0 :
        print(f"There is no fits fils in {DOINGDIR}")
        pass
    else : 
        print(f"Starting: {str(DOINGDIR.parts[-1])}")

        summary = yfu.make_summary(DOINGDIR/"*.fit*")
        print("len(summary):", len(summary))
        print("summary:", summary)
        #print(summary["file"][0])

DOINGDIR /mnt/Rdata/OBS_data/ccd_test_folder/120LACHESIS_LIGHT_-_2023-10-10_-_GSON300_STF-8300M_-_1bin
len(fits_in_dir) 20
Starting: 120LACHESIS_LIGHT_-_2023-10-10_-_GSON300_STF-8300M_-_1bin
All 53 keywords (guessed from /mnt/Rdata/OBS_data/ccd_test_folder/120LACHESIS_LIGHT_-_2023-10-10_-_GSON300_STF-8300M_-_1bin/120LACHESIS_LIGHT_R_2023-10-10-17-03-30_120sec_GSON300_STF-8300M_-19c_1bin.fits) will be loaded.
len(summary): 20
summary:                                                  file  filesize  SIMPLE  \
0   /mnt/Rdata/OBS_data/ccd_test_folder/120LACHESI...  16980480    True   
1   /mnt/Rdata/OBS_data/ccd_test_folder/120LACHESI...  16980480    True   
2   /mnt/Rdata/OBS_data/ccd_test_folder/120LACHESI...  16980480    True   
3   /mnt/Rdata/OBS_data/ccd_test_folder/120LACHESI...  16980480    True   
4   /mnt/Rdata/OBS_data/ccd_test_folder/120LACHESI...  16980480    True   
5   /mnt/Rdata/OBS_data/ccd_test_folder/120LACHESI...  16980480    True   
6   /mnt/Rdata/OBS_data/ccd_test_fold

In [5]:
summary = yfu.make_summary(DOINGDIR/"*.fit*")
print("len(summary):", len(summary))
print("summary:", summary)
#print(summary["file"][0])

All 53 keywords (guessed from /mnt/Rdata/OBS_data/ccd_test_folder/120LACHESIS_LIGHT_-_2023-10-10_-_GSON300_STF-8300M_-_1bin/120LACHESIS_LIGHT_R_2023-10-10-17-03-30_120sec_GSON300_STF-8300M_-19c_1bin.fits) will be loaded.
len(summary): 20
summary:                                                  file  filesize  SIMPLE  \
0   /mnt/Rdata/OBS_data/ccd_test_folder/120LACHESI...  16980480    True   
1   /mnt/Rdata/OBS_data/ccd_test_folder/120LACHESI...  16980480    True   
2   /mnt/Rdata/OBS_data/ccd_test_folder/120LACHESI...  16980480    True   
3   /mnt/Rdata/OBS_data/ccd_test_folder/120LACHESI...  16980480    True   
4   /mnt/Rdata/OBS_data/ccd_test_folder/120LACHESI...  16980480    True   
5   /mnt/Rdata/OBS_data/ccd_test_folder/120LACHESI...  16980480    True   
6   /mnt/Rdata/OBS_data/ccd_test_folder/120LACHESI...  16980480    True   
7   /mnt/Rdata/OBS_data/ccd_test_folder/120LACHESI...  16980480    True   
8   /mnt/Rdata/OBS_data/ccd_test_folder/120LACHESI...  16980480    True   
9  

In [6]:
df_light = summary.loc[summary["IMAGETYP"] == "LIGHT"].copy()
df_light = df_light.reset_index(drop=True)
print("df_light:\n{}".format(df_light))

df_light:
                                                 file  filesize  SIMPLE  \
0   /mnt/Rdata/OBS_data/ccd_test_folder/120LACHESI...  16980480    True   
1   /mnt/Rdata/OBS_data/ccd_test_folder/120LACHESI...  16980480    True   
2   /mnt/Rdata/OBS_data/ccd_test_folder/120LACHESI...  16980480    True   
3   /mnt/Rdata/OBS_data/ccd_test_folder/120LACHESI...  16980480    True   
4   /mnt/Rdata/OBS_data/ccd_test_folder/120LACHESI...  16980480    True   
5   /mnt/Rdata/OBS_data/ccd_test_folder/120LACHESI...  16980480    True   
6   /mnt/Rdata/OBS_data/ccd_test_folder/120LACHESI...  16980480    True   
7   /mnt/Rdata/OBS_data/ccd_test_folder/120LACHESI...  16980480    True   
8   /mnt/Rdata/OBS_data/ccd_test_folder/120LACHESI...  16980480    True   
9   /mnt/Rdata/OBS_data/ccd_test_folder/120LACHESI...  16980480    True   
10  /mnt/Rdata/OBS_data/ccd_test_folder/120LACHESI...  16980480    True   
11  /mnt/Rdata/OBS_data/ccd_test_folder/120LACHESI...  16980480    True   
12  /mnt/Rdata/

### Solving

In [7]:
#%%
#########################################
#checkPSolve
#########################################    
# 
# COMMENT scale: 2.90368 arcsec/pix     

# CD1_1   =  2.537000796571E-004 / CD matrix to convert (x,y) to (Ra, Dec)        
# CD1_2   = -2.549453735202E-005 / CD matrix to convert (x,y) to (Ra, Dec)        
# CD2_1   =  2.555377854783E-005 / CD matrix to convert (x,y) to (Ra, Dec)        
# CD2_2   =  2.536985194591E-004 / CD matrix to convert (x,y) to (Ra, Dec)        
# PLTSOLVD=                    T / Astrometric solved by ASTAP v2022.09.27.       
# COMMENT 7  Solved in 0.1 sec. Offset 0.0". Mount offset RA=-4.9", DEC=-19.2"

In [8]:
# #%%
# import _astro_utilities
# for _, row  in df_light.iterrows():
#     ASTAPSolver_status = False
#     ASTROMETRYSolver_status = False
#     fpath = Path(row["file"])
#     hdul = fits.open(fpath)
#     #ASTAP = _astro_utilities.checkASTAPPSolve(fpath)
#     SOLVE, ASTAP, LOCAL = _astro_utilities.checkPSolve(fpath)
#     print(SOLVE, ASTAP, LOCAL)

#     print(ASTAP)
#     if ASTAP :
#         print(f"{fpath.name} is solved by ASTAP")
#     else : 
#         print(f"{fpath.name} is solving now by ASTAP")
#         if 'PIXSCALE' in hdul[0].header:
#             PIXc = hdul[0].header['PIXSCALE']
#         else : 
#             PIXc = _astro_utilities.calPixScale(hdul[0].header['FOCALLEN'], hdul[0].header['XPIXSZ'])
#         print("PIXc : ", PIXc)
#         hdul.close()

#         solved = _astro_utilities.ASTAPSolver(fpath, 
#                                                 #str(SOLVEDDIR), 
#                                                 downsample = 2,
#                                                 pixscale = PIXc,
#                                                         )


In [9]:
#%%
import _astro_utilities
for _, row  in df_light.iterrows():
    ASTAPSolver_status = False
    ASTROMETRYSolver_status = False
    fpath = Path(row["file"])
    hdul = fits.open(fpath)
    #ASTAP = _astro_utilities.checkASTAPPSolve(fpath)
    SOLVE, ASTAP, LOCAL = _astro_utilities.checkPSolve(fpath)
    print(SOLVE, ASTAP, LOCAL)

    print(LOCAL)
    if LOCAL :
        print(f"{fpath.name} is solved by LOCAL")
    else : 
        print(f"{fpath.name} is solving now by LOCAL")
        if 'PIXSCALE' in hdul[0].header:
            PIXc = hdul[0].header['PIXSCALE']
        else : 
            PIXc = _astro_utilities.calPixScale(hdul[0].header['FOCALLEN'], hdul[0].header['XPIXSZ'])
        print("PIXc : ", PIXc)
        hdul.close()

        solved = _astro_utilities.LOCALPSolver(fpath, 
                                                #str(SOLVEDDIR), 
                                                downsample = 2,
                                                pixscale = PIXc,
                                                        )

False False False
False
120LACHESIS_LIGHT_R_2023-10-10-17-03-30_120sec_GSON300_STF-8300M_-19c_1bin.fits is solving now by LOCAL
PIXc :  0.9281925000000001
downsample:  2
pixscale: 0.928, L: 0.900, U: 0.956
b'Reading input file 1 of 1: "/mnt/Rdata/OBS_data/ccd_test_folder/120LACHESIS_LIGHT_-_2023-10-10_-_GSON300_STF-8300M_-_1bin/120LACHESIS_LIGHT_R_2023-10-10-17-03-30_120sec_GSON300_STF-8300M_-19c_1bin.fits"...\nExtracting sources...\nDownsampling by 2...\nsimplexy: found 1612 sources.\nSolving...\nWarning: encountered two index files with the same INDEXID = 2020033107 and HEALPIX = 11: "/mnt/8TB1/OBS_data/astrometry/data/UCAC3/index-ucac3-7-11.fits" and "/mnt/8TB1/OBS_data/astrometry/data/UCAC4/index-ucac4-7-11.fits".  Keeping both.\nWarning: encountered two index files with the same INDEXID = 2020033107 and HEALPIX = 10: "/mnt/8TB1/OBS_data/astrometry/data/UCAC3/index-ucac3-7-10.fits" and "/mnt/8TB1/OBS_data/astrometry/data/UCAC4/index-ucac4-7-10.fits".  Keeping both.\nWarning: encoun

In [ ]:
#%%
import _astro_utilities
for _, row  in df_light.iterrows():
    ASTAPSolver_status = False
    ASTROMETRYSolver_status = False
    fpath = Path(row["file"])
    hdul = fits.open(fpath)
    #ASTAP = _astro_utilities.checkASTAPPSolve(fpath)
    SOLVE, ASTAP, LOCAL = _astro_utilities.checkPSolve(fpath)
    print(SOLVE, ASTAP, LOCAL)

    print(LOCAL)
    if LOCAL :
        print(f"{fpath.name} is solved by LOCAL")
    else : 
        print(f"{fpath.name} is solving now by LOCAL")
        if 'PIXSCALE' in hdul[0].header:
            PIXc = hdul[0].header['PIXSCALE']
        else : 
            PIXc = _astro_utilities.calPixScale(hdul[0].header['FOCALLEN'], hdul[0].header['XPIXSZ'])
        print("PIXc : ", PIXc)
        hdul.close()

        solved = _astro_utilities.LOCALPSolver(fpath, 
                                                #str(SOLVEDDIR), 
                                                downsample = 2,
                                                pixscale = PIXc,
                                                        )
    SOLVE, ASTAP, LOCAL = _astro_utilities.checkPSolve(fpath)
    print(SOLVE, ASTAP, LOCAL)
    print(ASTAP)
    if ASTAP :
        print(f"{fpath.name} is solved by ASTAP")
    else : 
        print(f"{fpath.name} is solving now by ASTAP")
        if 'PIXSCALE' in hdul[0].header:
            PIXc = hdul[0].header['PIXSCALE']
        else : 
            PIXc = _astro_utilities.calPixScale(hdul[0].header['FOCALLEN'], hdul[0].header['XPIXSZ'])
        print("PIXc : ", PIXc)
        hdul.close()

        solved = _astro_utilities.ASTAPSolver(fpath, 
                                                #str(SOLVEDDIR), 
                                                downsample = 2,
                                                pixscale = PIXc,
                                                        )

True
True
True
True
True
